# Lab 3 最小 RAG：把「檢索」和「生成」縫起來（完整版）

上午的 Lab 1 你做了**檢索**——把文件變成向量，找出最相關的那幾篇。
下午的 Lab 2 你做了**生成**——把雲端 LLM 叫起來回答問題，最後手工把「資料 ＋ 問題」一起送過去。

Lab 2 結尾留了一個問題：**如果有 1000 份財報，總不能全部貼進 prompt 吧？**

這份 notebook 就是答案：**先用檢索挑出該送的那幾段，再送給 LLM 生成。** 這就是 RAG。
> **完整版**：所有答案都在，關鍵行有 `💡` 說明為什麼這樣寫。


In [ ]:
# 📦 先跑這一格：一次裝齊今天要用的套件（指定版本，避免學校電腦裝到不相容的舊版；裝不起來看 README）
!pip install -q langchain-ollama==1.0.1 langchain-community==0.4.2 langchain-text-splitters==1.1.2 faiss-cpu==1.14.2 numpy==1.26.4 requests

## 🔧 第 0 步：環境自我檢查（四格，跑一格看一個燈）

四個燈全綠再往下走。有紅燈先看 README。

In [2]:
# ✅ 檢查 1：本機 Ollama 服務通不通（今天的 embedding 要靠它，這個紅燈會擋住整個 Lab）
import requests

try:
    r = requests.get("http://localhost:11434/api/tags", timeout=3)
    print("✅ 本機 Ollama 服務通了")
except Exception:
    print("❌ 連不上本機 Ollama（http://localhost:11434）")
    print("   → 開一次 Ollama App，或在終端機下 ollama serve")

✅ 本機 Ollama 服務通了


In [3]:
# ✅ 檢查 2：設定 API key（今天呼叫雲端的「門票」，沒有它整個 Lab 做不下去）
import os

# 👇 同學：把引號中間換成老師給你的 OLLAMA key（整段貼進去，前後別留空白）
os.environ["OLLAMA_API_KEY"] = "在這裡貼上你的 OLLAMA key"

key = os.environ.get("OLLAMA_API_KEY")
if key and key != "在這裡貼上你的 OLLAMA key":
    print("✅ OLLAMA_API_KEY 已設定（長度", len(key), "個字元）")
else:
    print("❌ 還沒貼 key → 把上面那行引號中間換成你的 OLLAMA key，再重跑這一格")

# 🔒 只印長度、不印 key 本身——key 等於你帳號的鑰匙。
#    貼了 key 的 notebook 別上傳 GitHub、別傳給別人、別截圖給人看。

✅ OLLAMA_API_KEY 已設定（長度 57 個字元）


In [4]:
# ✅ 檢查 3：四個套件裝了沒
try:
    from langchain_ollama import OllamaEmbeddings, ChatOllama
    from langchain_community.vectorstores import FAISS
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    from langchain_core.prompts import ChatPromptTemplate
    print("✅ 四個套件都在")
except Exception as e:
    print("❌ 套件缺東西 →", e)
    print("   → 終端機下：pip install -r requirements.txt")

/home/barai/.local/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


/tmp/ipykernel_603347/3310279168.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


✅ 四個套件都在


In [5]:
# ✅ 檢查 4：embedding 模型 bge-m3 拉了沒（1.2GB，課前就要拉好）
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=3)
    names = []
    for m in r.json()["models"]:
        names.append(m["name"])

    if "bge-m3:latest" in names:
        print("✅ bge-m3 已備妥")
    else:
        print("❌ 還沒拉 bge-m3 → 終端機下：ollama pull bge-m3")
    print("   本機現有模型：", names)
except Exception:
    print("⚠️ 本機 Ollama 沒通，無法確認（同檢查 1）")

✅ bge-m3 已備妥
   本機現有模型： ['bge-m3:latest', 'nomic-embed-text:latest', 'gemma4:26b-a4b-it-q4_K_M', 'gemma4:26b', 'gemma4:12b', 'gpt-oss:20b', 'granite4:small-h', 'granite4:micro', 'gemma3:27b', 'gemma3:12b', 'llama3.2:3b', 'llama3.1:8b', 'mistral-small3.2:24b', 'mistral-nemo:12b']


---
## A・接上兩個模型

RAG 要用到**兩種**模型，而且跑在**兩個地方**：

| | 模型 | 跑在哪 | 做什麼 |
|---|---|---|---|
| `emb` | `bge-m3` | **你的電腦**（不花錢） | 把文字變成向量 |
| `llm` | `gemma4:cloud` | **雲端**（要 key、會扣額度） | 看著資料寫出答案 |

把**文字**變成**一排數字**（`bge-m3` 給的是 1024 個），這件事叫 **embedding**，那排數字叫**向量**；重點只有一句：**意思像的，數字就接近**——就是課堂白板那張「哪兩部是同一掛」的圖。下面幾格就把這句話**量給你看**。

### A1・建立兩個模型物件

**預期輸出：** 兩行，各自說明跑在哪。（這格只是建立連線物件，還沒真的呼叫模型。）

In [6]:
import os
from langchain_ollama import OllamaEmbeddings, ChatOllama

# (1) embedding —— 跑本機，不用 key
emb = OllamaEmbeddings(model="bge-m3")

# (2) 生成 —— 跑雲端，要帶 key
llm = ChatOllama(
    model="gemma4:cloud",          # 雲端模型要帶 cloud tag；漏了它會去找「本機」的同名模型 → model not found
    base_url="https://ollama.com", # 打到雲端（不寫這行就是打你自己的電腦）
    client_kwargs={"headers": {"Authorization": f"Bearer {os.environ['OLLAMA_API_KEY']}"}},
)

print("embedding：", emb.model, "→ 跑你的電腦")
print("生成    ：", llm.model, "→ 跑雲端")

embedding： bge-m3 → 跑你的電腦
生成    ： gemma4:cloud → 跑雲端


### A2・先看一眼：embedding 到底長什麼樣

把「升息」兩個字丟給 `emb`，拿回來的是什麼？

**預期輸出：** 一個長度 **1024** 的數字清單。只印長度和前 5 個——全部印出來是一整頁數字，看了也沒意義。

In [7]:
v = emb.embed_query("升息")

print("型別：", type(v))
print("長度：", len(v))          # 💡 1024 個數字 = bge-m3 這個模型的「維度」。換一個 embedding 模型，維度就不一樣
print("前 5 個數字：", v[:5])
# 💡 這 1024 個數字就是「升息」的語意座標。單看它毫無意義——
#    它的意義只存在於「跟別的向量比」的時候。下一格就來比

型別： <class 'list'>
長度： 1024
前 5 個數字： [-0.02351814, 0.015823388, -0.023441179, 0.012181802, -0.039419852]


---
## B・語意近不近，用數字量給你看

上午的 TF-IDF 比的是「**字面**有沒有出現」，所以「綠能」和「再生能源」比出來是 0 分——一個字都沒重疊。

embedding 比的是「**意思**像不像」。真的嗎？量給你看。

### B1・算兩個句子的相似度

`cosine_similarity`（餘弦相似度）：兩個向量方向越接近，值越接近 1。**這是上午 Lab 1 排名時用過的同一把尺。**

**預期輸出：** 一個 0 到 1 之間的數字。

In [8]:
import numpy as np

def cos(a, b):
    a = np.array(a)
    b = np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
    # 💡 這就是上午算 cosine 的公式：內積 ÷ 兩個向量的長度。
    #    上午餵它 TF-IDF 向量，現在餵它 embedding 向量——尺沒變，只換了向量怎麼來的

va = emb.embed_query("公司本季毛利率是 38%")
vb = emb.embed_query("這家公司的獲利能力如何？")

print("相似度：", round(float(cos(va, vb)), 3))
# 💡 這兩句話「一個字都沒重疊」（毛利率 vs 獲利能力）——TF-IDF 會給 0 分，embedding 給高分

相似度： 0.677


### B2・三組對照：該近的近、該遠的遠

每組都是「**語意相近的一對**」對上「**完全無關的一對**」（拿珍珠奶茶當對照組）。

**預期輸出：** 每組「近」的數字，要明顯大於「遠」的。

In [9]:
milk_tea = "這家店的珍珠奶茶半糖去冰最好喝"

pairs = [
    ("央行今天決議升息半碼", "利率調高會讓房貸族壓力變大"),
    ("公司本季毛利率是 38%", "這家公司的獲利能力如何？"),
    ("政府大力推動綠能發電", "再生能源是未來趨勢"),
]

v_milk_tea = emb.embed_query(milk_tea)

for s1, s2 in pairs:
    v1 = emb.embed_query(s1)
    v2 = emb.embed_query(s2)
    near = round(float(cos(v1, v2)), 3)
    far = round(float(cos(v1, v_milk_tea)), 3)
    print("近：", near, "  遠：", far, "  ←", s1)

近： 0.595   遠： 0.439   ← 央行今天決議升息半碼


近： 0.677   遠： 0.443   ← 公司本季毛利率是 38%


近： 0.605   遠： 0.335   ← 政府大力推動綠能發電


### 🔑 上午那個「比不到」的坑，補起來了

Lab 1 的小作業裡，你用 TF-IDF 搜「綠能」，搜不到寫「再生能源」的那篇——**字面沒重疊，分數就是 0**。

同一對句子，embedding 給了高分。

> **這就是 RAG 為什麼不用 TF-IDF 做檢索、改用 embedding：抓得到「意思一樣但用字不同」的內容。**

---
## C・切塊（chunking）

真實文件動輒幾十頁，不可能整篇塞給 LLM。所以先把長文件剪成一段一段（叫 **chunking／切塊**），檢索時只撈出**跟問題相關的那幾段**；讓相鄰兩塊頭尾多抄一段，叫 **`chunk_overlap`**。

⚠️ 多抄那一段不是浪費——**一句話被從中間剪斷，兩塊誰都答不出那題**。C4 那格會把重疊印出來給你看。

### C1・先用 3 句話，看看切塊回傳什麼

**預期輸出：** 切出 3 塊；最後一行會顯示 `Document(...)`——這是 LangChain 裝文字的盒子。

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

three_lines = [
    "宏圖飲料 2026 Q1 營收 12.5 億元，毛利率 38%，EPS 2.1 元。",
    "宏圖飲料 2026 Q1 法說會：預期第二季毛利率略降至 36%。",
    "宏圖飲料 2026 Q1 股利政策：董事會尚未決議。",
]

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
# 💡 chunk_size=300：每塊最多約 300 字
# 💡 chunk_overlap=50：相鄰兩塊「頭尾多抄 50 字」，免得把一句話從中間剪斷、意思接不上

small = splitter.create_documents(three_lines)

print("切出", len(small), "塊")
small[0]      # 💡 最後一行不寫 print，Jupyter 會直接把它的「原始長相」顯示出來

切出 3 塊


Document(metadata={}, page_content='宏圖飲料 2026 Q1 營收 12.5 億元，毛利率 38%，EPS 2.1 元。')

### C2・換成真的文件

`data/` 資料夾裡有三份**虛構的**宏圖飲料財報（財報摘要、法說會逐字稿、股利與風險揭露）。

**預期輸出：** 讀到 3 個檔，總共一千多字。

In [11]:
import glob

paths = sorted(glob.glob("data/*.txt"))
# 💡 glob = 「把符合 data/ 底下所有 .txt 的檔名抓成清單」；sorted 是為了讓每個人跑出來的順序一樣

raw = []
for p in paths:
    with open(p, encoding="utf-8") as f:
        raw.append(f.read())
    print("讀到：", p)

total = 0
for text in raw:
    total = total + len(text)
print("總字數：", total)

讀到： data/宏圖飲料_2026Q1_法說會逐字稿.txt
讀到： data/宏圖飲料_2026Q1_股利與風險揭露.txt
讀到： data/宏圖飲料_2026Q1_財報摘要.txt
總字數： 1674


### C3・切下去

同一個 `splitter`，這次餵它三份長文件。

**預期輸出：** 切出 **7 塊**，每塊 200~300 字。

In [12]:
chunks = splitter.create_documents(raw)     # 💡 跟 C1 同一個方法，只是這次餵的是長文件

print("切出", len(chunks), "塊")
print()
for i, ch in enumerate(chunks):     # 💡 enumerate = 一邊跑迴圈一邊給編號
    print("第", i, "塊（", len(ch.page_content), "字）：", ch.page_content[:30], "...")
    # 💡 .page_content 就是從 Document 這個盒子裡把「文字」拿出來——跟 Lab 2 的 .content 同一個道理

切出 7 塊

第 0 塊（ 273 字）： 【教學用虛構文件】宏圖飲料 2026 年第一季法人說明會 逐 ...
第 1 塊（ 291 字）： 分析師提問：關於售價，公司有沒有調漲的規劃？

財務長：目前 ...
第 2 塊（ 218 字）： 【教學用虛構文件】宏圖飲料 2026 年第一季 股利政策與風 ...
第 3 塊（ 289 字）： 二、營運風險
1. 原物料價格風險：濃縮果汁、糖、紙盒包材價 ...
第 4 塊（ 260 字）： 【教學用虛構文件】宏圖飲料股份有限公司 2026 年第一季財 ...
第 5 塊（ 267 字）： 二、通路與產品結構
本季營收依通路別劃分：便利商店占 46% ...
第 6 塊（ 89 字）： 四、資本支出
本季資本支出為 0.7 億元，主要用於桃園二廠 ...


### C4・`chunk_overlap` 到底做了什麼？看給你

第 0 塊的**尾巴**，跟第 1 塊的**開頭**，應該長得一樣。

**預期輸出：** 兩行印出來的字會有一段重疊。

In [13]:
print("第 0 塊的最後 40 字：")
print("  ", chunks[0].page_content[-40:])
print()
print("第 1 塊的最前 40 字：")
print("  ", chunks[1].page_content[:40])
# 💡 看到重疊了嗎？這就是 overlap=50 在做的事。
#    如果剛好在「財務長：預期第二季毛利率會降到 36%」中間剪一刀，兩塊誰都答不出這題。
#    頭尾多抄一段，就能避免「答案被剪斷」的慘案

第 0 塊的最後 40 字：
   整產品組合，來部分抵銷成本壓力。

分析師提問：關於售價，公司有沒有調漲的規劃？

第 1 塊的最前 40 字：
   分析師提問：關於售價，公司有沒有調漲的規劃？

財務長：目前沒有全面調漲的計畫。


---
## D・建向量庫

把每一塊都算成向量，存進一個「**能秒找最相似向量**」的資料庫（我們用 `FAISS`）。這一步就是 RAG 的「**建索引**」——**做一次就好**；之後丟一個問題進去，它就能秒回「**庫裡哪幾塊跟你最像**」。

### D1・把 7 塊都變成向量、存起來

**預期輸出：** 向量庫裡有 7 塊。（這格會呼叫本機 embedding 算 7 次，要跑幾秒。）

In [14]:
from langchain_community.vectorstores import FAISS

vs = FAISS.from_documents(chunks, emb)
# 💡 這一行做兩件事：① 把 7 塊文字每塊都送給 emb 算成 1024 維向量 ② 把向量存進 FAISS
# 💡 FAISS 是 Facebook 出的向量庫，跑在你的記憶體裡，很快。
#    它只做一件事：給它一個向量，秒回「庫裡哪幾個跟你最像」

print("向量庫塊數：", vs.index.ntotal)

向量庫塊數： 7


---
## E・檢索：撈出跟問題最相關的 k 塊

**這一步還沒有生成任何答案**——只是「把相關的紙條抽出來」。它就是上午 Lab 1 做的事，只是把 TF-IDF 換成 embedding。

### E1・撈回來的東西，原本長什麼樣

**預期輸出：** 一個 `Document` 物件——跟 C1 看到的同一種盒子。

In [15]:
q = "這季毛利率多少？"

hits = vs.similarity_search(q, k=2)
# 💡 k=2：撈最相關的 2 塊

print("拿回來的是：", type(hits), "，裡面有", len(hits), "個東西")
print()
hits[0]      # 💡 直接顯示第一名的「原始長相」——又是那個 Document 盒子

拿回來的是： <class 'list'> ，裡面有 2 個東西



Document(id='26df3f4f-88ee-42ce-b7e8-917ee380d62c', metadata={}, page_content='【教學用虛構文件】宏圖飲料 2026 年第一季法人說明會 逐字稿摘錄\n（本檔所有公司、人名、數字皆為虛構教學範例，非真實行情，請勿引用）\n\n主持人：接下來請財務長說明第二季展望。\n\n財務長：謝謝。第一季我們的毛利率是 38%，表現符合預期。不過要提醒各位，第二季開始，我們主要原物料當中的濃縮果汁與糖，採購價格都出現明顯上漲，紙盒包材也隨著紙漿報價調升。在售價不調整的前提下，我們預期第二季毛利率會略為下降到 36% 左右。公司會透過提高自有工廠的產能利用率、以及調整產品組合，來部分抵銷成本壓力。\n\n分析師提問：關於售價，公司有沒有調漲的規劃？')

### E2・把文字從盒子裡挑出來

跟 Lab 2 一樣：拿回來的是包裹，要自己挑出裡面的文字。

| | 拿回來的包裹 | 文字在哪 |
|---|---|---|
| Lab 2 | `AIMessage` | `.content` |
| Lab 3 | `Document` | `.page_content` |

**預期輸出：** 兩塊財報原文。第 1 相關那塊裡面應該找得到「毛利率 38%」。

In [16]:
for i, d in enumerate(hits, 1):     # 💡 enumerate(hits, 1)：編號從 1 開始數
    print("[第", i, "相關]")
    print(d.page_content)
    print("-" * 40)

[第 1 相關]
【教學用虛構文件】宏圖飲料 2026 年第一季法人說明會 逐字稿摘錄
（本檔所有公司、人名、數字皆為虛構教學範例，非真實行情，請勿引用）

主持人：接下來請財務長說明第二季展望。

財務長：謝謝。第一季我們的毛利率是 38%，表現符合預期。不過要提醒各位，第二季開始，我們主要原物料當中的濃縮果汁與糖，採購價格都出現明顯上漲，紙盒包材也隨著紙漿報價調升。在售價不調整的前提下，我們預期第二季毛利率會略為下降到 36% 左右。公司會透過提高自有工廠的產能利用率、以及調整產品組合，來部分抵銷成本壓力。

分析師提問：關於售價，公司有沒有調漲的規劃？
----------------------------------------
[第 2 相關]
二、通路與產品結構
本季營收依通路別劃分：便利商店占 46%、量販與超市占 27%、餐飲通路占 18%、電商與其他占 9%。產品結構方面，茶飲類占營收 52%，果汁類占 23%，咖啡類占 15%，其他（氣泡飲、機能飲料）占 10%。管理層指出，咖啡類毛利率高於公司平均，是未來三年主要擴張方向。

三、費用與現金
本季營業費用為 2.8 億元，占營收 22.4%，其中行銷費用 1.1 億元，較去年同期增加 15%，主要投入於新品上市的通路陳列與社群行銷。期末現金與約當現金為 6.3 億元，短期借款 1.2 億元，負債比率 34%。
----------------------------------------


### E3・換你出題：問一個「文件裡沒有這些字」的問題

embedding 抓的是**意思**，不是**字面**。所以你可以用文件裡**完全沒出現過的說法**去問。

自己想一個問題填進去試試看——例如「投資這家公司要小心什麼？」（文件裡沒有「投資」「小心」這些字）。

**預期輸出：** 它照樣撈回意思相關的段落。**上午的 TF-IDF 做不到這件事**（字面沒重疊 → 分數 0）。

In [17]:
q2 = "投資這家公司要小心什麼？"      # 💡 文件裡沒有「投資」「小心」這些字，但它會撈到〈營運風險〉

for d in vs.similarity_search(q2, k=2):
    print("→", d.page_content[:60])
    print()

→ 二、營運風險
1. 原物料價格風險：濃縮果汁、糖、紙盒包材價格波動，將直接影響毛利率。公司採取分批採購與部分長約鎖價因應

→ 【教學用虛構文件】宏圖飲料 2026 年第一季 股利政策與風險揭露
（本檔所有公司、人名、數字皆為虛構教學範例，非真實行



### 📝 小作業 A

多試幾個問法，找出**一個會撈歪的問題**。

例如「**這家公司有什麼地雷？**」——「地雷」和「風險」意思很近，但它撈得到〈營運風險〉嗎？

<details><summary>📖 做完再看：參考解（參考解不只一種）</summary>

```python
for d in vs.similarity_search("這家公司有什麼地雷？", k=2):
    print("→", d.page_content[:60])
    print()
```

**它撈歪了**——跑去撈法說會，沒撈到〈營運風險〉。

但換成「投資這家公司要小心什麼？」就撈對了。

> **embedding 抓語意，但不保證每次都抓對。** 這正是下面 G 段和「k 值」要處理的事：
> **RAG 答錯的時候，多半錯在檢索這一站，不是模型笨。** 先把 `hits` 印出來看撈到什麼，再怪模型。
</details>

In [18]:
# 📝 小作業 A 參考解（參考解不只一種）

# 找一個「撈歪」的問法：語意接近但檢索跑偏
for d in vs.similarity_search("這家公司有什麼地雷？", k=2):
    print("→", d.page_content[:60])
    print()

# 💡 它撈歪了——跑去撈法說會，沒撈到〈營運風險〉；換成「投資這家公司要小心什麼？」就撈對了。
# 結論：embedding 抓語意，但不保證每次都抓對。RAG 答錯多半錯在「檢索」這一站，不是模型笨——
#       先把撈到的塊印出來看，再怪模型。

→ 【教學用虛構文件】宏圖飲料 2026 年第一季法人說明會 逐字稿摘錄
（本檔所有公司、人名、數字皆為虛構教學範例，非真實

→ 分析師提問：關於售價，公司有沒有調漲的規劃？

財務長：目前沒有全面調漲的計畫。我們評估過，主力茶飲產品在便利商店通路的



---
## F・組 Prompt + 生成：縫合的那一刀

把檢索到的 k 塊文字，連同問題一起餵給 LLM，叫它「**只根據這些資料回答**」。

### F1・RAG 的黃金模板

模板有三個部分：**護欄指令** ＋ **【資料】** ＋ **【問題】**。

**預期輸出：** 印出模板長相，`{ctx}` 和 `{q}` 是待填的空格。

In [19]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    "你是嚴謹的助理。只根據下面提供的資料回答問題，"
    "若資料中沒有答案就說「資料中查無此資訊」，不要自己編。\n\n"
    "【資料】\n{ctx}\n\n【問題】{q}\n\n請用繁體中文回答："
)
# 💡 「只根據資料回答、查不到就說查無」這句就是**護欄**。
#    Lab 2 最後你手工寫過一次類似的句子——那時資料是你自己貼的，現在由檢索自動撈

print(prompt.format(ctx="（這裡會塞檢索到的文字）", q="（這裡會塞問題）"))

Human: 你是嚴謹的助理。只根據下面提供的資料回答問題，若資料中沒有答案就說「資料中查無此資訊」，不要自己編。

【資料】
（這裡會塞檢索到的文字）

【問題】（這裡會塞問題）

請用繁體中文回答：


### F2・先看一眼：要塞進【資料】欄的東西長怎樣

把檢索到的 2 塊文字，用換行接成一整段。

**預期輸出：** 兩塊財報原文接在一起——**這就是等一下要餵給 LLM 的「資料」**。

In [20]:
texts = []
for d in hits:
    texts.append(d.page_content)

ctx = "\n".join(texts)
# 💡 把 2 塊文字用換行接成一整段。這一段就是要塞進 prompt【資料】欄的東西

print(ctx)

【教學用虛構文件】宏圖飲料 2026 年第一季法人說明會 逐字稿摘錄
（本檔所有公司、人名、數字皆為虛構教學範例，非真實行情，請勿引用）

主持人：接下來請財務長說明第二季展望。

財務長：謝謝。第一季我們的毛利率是 38%，表現符合預期。不過要提醒各位，第二季開始，我們主要原物料當中的濃縮果汁與糖，採購價格都出現明顯上漲，紙盒包材也隨著紙漿報價調升。在售價不調整的前提下，我們預期第二季毛利率會略為下降到 36% 左右。公司會透過提高自有工廠的產能利用率、以及調整產品組合，來部分抵銷成本壓力。

分析師提問：關於售價，公司有沒有調漲的規劃？
二、通路與產品結構
本季營收依通路別劃分：便利商店占 46%、量販與超市占 27%、餐飲通路占 18%、電商與其他占 9%。產品結構方面，茶飲類占營收 52%，果汁類占 23%，咖啡類占 15%，其他（氣泡飲、機能飲料）占 10%。管理層指出，咖啡類毛利率高於公司平均，是未來三年主要擴張方向。

三、費用與現金
本季營業費用為 2.8 億元，占營收 22.4%，其中行銷費用 1.1 億元，較去年同期增加 15%，主要投入於新品上市的通路陳列與社群行銷。期末現金與約當現金為 6.3 億元，短期借款 1.2 億元，負債比率 34%。


### F3・縫合的那一刀

`format`（F1 用過）回的是**一個字串**，給人看的。
`format_messages` 回的是**一串訊息物件**（帶著「誰說的」這個身分）——**chat 模型吃的是這種格式**。

**預期輸出：** 它照著財報答出 **38%**。

In [21]:
messages = prompt.format_messages(ctx=ctx, q=q)
# 💡 format vs format_messages：
#    format          → 回「一個字串」（人看的）
#    format_messages → 回「一串訊息物件」（chat 模型吃的格式，帶著 Human/AI 的身分標籤）

resp = llm.invoke(messages)          # 💡 這一步打雲端、會扣額度

print(resp.content)

第一季的毛利率是 38%。


### H1・`ask_rag()`

**預期輸出：** 問「第二季毛利率」→ 答 36% 左右。

In [22]:
def ask_rag(question, k=2):
    found = vs.similarity_search(question, k=k)     # ① 撈
    parts = []
    for d in found:
        parts.append(d.page_content)
    data = "\n".join(parts)                          # ② 拼
    return llm.invoke(prompt.format_messages(ctx=data, q=question)).content   # ③ 送
    # 💡 這三步就是 RAG 的全部：撈 → 拼 → 送

print(ask_rag("第二季毛利率會怎樣？"))

預期第二季毛利率會略為下降到 36% 左右。


---

## 更深一層

上面做到 `ask_rag()`，你已經有一個**會查資料再回答**的最小 RAG 了。

以下再往下看兩件事：**護欄擋得住多少**、**為什麼不乾脆把資料全部貼進去**。每一格都有完整的輸出和說明，照著跑就好。


---
## G・那句護欄，擋得住多少？

F1 的模板裡有一句護欄：「**資料中沒有答案就說查無此資訊，不要自己編。**」

它到底有沒有用？我們**用同一份資料、同一件事，換兩種問法**，親手測它的底線。

### G1・先看護欄有用的時候

財報只寫了 **2026 Q1 營收 12.5 億、年增 8%**。它**從來沒寫 2025 年的營收是多少**。

**預期輸出：** 它老實回「資料中查無此資訊」。護欄成功擋下。

In [23]:
trap_q = "宏圖飲料 2025 年第一季的營收是多少？"

trap_hits = vs.similarity_search(trap_q, k=2)
trap_texts = []
for d in trap_hits:
    trap_texts.append(d.page_content)
trap_ctx = "\n".join(trap_texts)

print("【護欄・客氣地問】")
print(llm.invoke(prompt.format_messages(ctx=trap_ctx, q=trap_q)).content)

【護欄・客氣地問】


資料中查無此資訊


### G2・同一份資料、同一個護欄，只把問法改一下

這次在問題後面加五個字：「**請算給我看。**」

**護欄一個字都沒改。**

**預期輸出：** 它**算給你看**——12.5 ÷ 1.08 ≈ 11.57 億元，附上算式，語氣篤定。

In [24]:
push_q = "宏圖飲料 2025 年第一季的營收大約是多少？請算給我看。"
# 💡 資料一樣、prompt 模板一樣（護欄還在）——只有問法變了

push_hits = vs.similarity_search(push_q, k=2)
push_texts = []
for d in push_hits:
    push_texts.append(d.page_content)
push_ctx = "\n".join(push_texts)

print("【護欄・叫它算】")
print(llm.invoke(prompt.format_messages(ctx=push_ctx, q=push_q)).content)

【護欄・叫它算】


2026 年第一季合併營收為新台幣 12.5 億元，且較去年同期成長 8%。

計算方式如下：
12.5 億 ÷ (1 + 8%) ≈ 11.57 億元

宏圖飲料 2025 年第一季的營收大約是新台幣 11.57 億元。


### 🔑 兩件事，一次看懂

**第一：這就是幻覺的真實長相。**

它**沒有瞎編**——它從「12.5 億、年增 8%」**合理推導**出一個數字，算式甚至是對的。

問題是：**財報從來沒說過 2025 Q1 的營收是 11.57 億。** 那個 8% 是四捨五入過的年增率，回推出來的數字本身就有誤差；而且這個數字是**模型算的，不是財報寫的**。

> 在金融場景，**「看起來很有道理的推導數字」比「明顯的胡說八道」危險得多**——因為它不會被一眼看穿。你敢把它直接貼進給主管的報告嗎？

**第二：護欄不是鐵門，是柵欄。**

同一句護欄，客氣地問 → 擋住了；叫它算 → **它就算了**。

> **RAG ＋ 護欄能「降低」幻覺，不能「消除」幻覺。**
>
> 所以真實的金融應用，最後一定還有一道**人工複核**：AI 給的每個數字，要能指回原文哪一句。指不回去的，就不能用。

---
## G2・護欄被攻破了，那要怎麼補？

先問一個問題：**它為什麼不聽話？**

**不是模型叛逆，是我們的規則有漏洞。** 護欄說的是「若資料中**沒有答案**就說查無」——
但模型心裡想的是：「營收 12.5 億、年增 8% 都給我了，**這題我算得出來**，那不就等於『有答案』嗎？」

> **「沒有答案」這四個字有解釋空間，它選了對自己有利的那一邊。**
> 這是 prompt 設計的通病：**你以為講清楚了，其實只講了你想到的那一半。**

下面試兩種補法。**兩次都用同一題（「請算給我看」那題）、同一份資料。**

### G2-1・補法一：把漏洞明講

指令裡直接補上「**即使可以從資料推算出來，也不准推算**」。**沒有給任何範例。**

**預期輸出：** `資料中查無此資訊`——擋住了。

In [25]:
guard2 = f"""你是嚴謹的助理。只根據下面提供的資料回答問題。
資料中沒有寫的，一律回「資料中查無此資訊」——即使可以從資料推算出來，也不准推算。

【資料】
{push_ctx}

【問題】{push_q}

請用繁體中文回答："""
# 💡 跟原本的護欄比，只多了「即使可以推算出來，也不准推算」這半句——把它鑽的那個漏洞堵死

print("【補法一・把漏洞明講】")
print(llm.invoke(guard2).content)

【補法一・把漏洞明講】


資料中查無此資訊


### G2-2・補法二：做一次給它看（one-shot・單樣本）

這次**指令一個字都不改**（還是原本那句被攻破的護欄），只在前面**附一個示範**：

> 「遇到這種『叫你算』的題目，就回查無此資訊。」

這就是上午提示工程講的 **one-shot（給一個範例）**。

**預期輸出：** 一樣擋住。

In [26]:
one_shot = f"""你是嚴謹的助理。只根據下面提供的資料回答問題，
若資料中沒有答案就說「資料中查無此資訊」，不要自己編。

【示範】
資料：本季（2026 Q1）營收 12.5 億元，較去年同期成長 8%。
問題：去年同期的營收是多少？請算給我看。
回答：資料中查無此資訊。

【資料】
{push_ctx}

【問題】{push_q}

請用繁體中文回答："""
# 💡 護欄那句話「不要自己編」原封不動——差別只在多了一個【示範】

print("【補法二・給它一個示範（one-shot）】")
print(llm.invoke(one_shot).content)

【補法二・給它一個示範（one-shot）】


資料中查無此資訊。


### 🔑 兩條路都通，但走的是不同的路

| Prompt 設計 | 擋住了嗎 |
|---|---|
| **原護欄**（G1/G2 那個） | ❌ 被攻破，算出 11.57 億 |
| **補法一**・把漏洞明講（沒給範例） | ✅ 擋住 |
| **補法二**・給一個示範（指令沒改） | ✅ 擋住 |

- **補法一 ＝ 把話講得更死。** 便宜、prompt 短。
- **補法二 ＝ 做一次給它看。** 你**不必窮舉**「推算、估計、換算、推估…」每一種說法（**列不完，總會漏**）——**示範一次，它自己會歸納。**

> **職場上的順序：先寫清楚指令 → 怎麼調都不穩，再上示範。**（＝上午提示工程講的「先 zero-shot，不夠穩再補範例」。）

### G2-3・那給「更多個」示範（few-shot）會更好嗎？

下面這個版本給了 **3 個示範**（該答的照答、資料沒寫的回查無、叫你推估的也回查無）。

**盯著看兩件事：** ① 陷阱題擋住了嗎？ ② **該答的那題，回答變成什麼樣子？**

In [27]:
def few_shot_ask(question):
    found = vs.similarity_search(question, k=2)
    parts = []
    for d in found:
        parts.append(d.page_content)
    data = "\n".join(parts)

    p = f"""你是嚴謹的助理。只根據下面提供的資料回答問題。
資料中沒有寫的，一律回「資料中查無此資訊」——即使可以從資料推算出來，也不准推算。

【示範1】
資料：本季（2026 Q1）營收 12.5 億元，較去年同期成長 8%。
問題：去年同期的營收是多少？請算給我看。
回答：資料中查無此資訊。

【示範2】
資料：本季毛利率 38%。
問題：本季毛利率是多少？
回答：38%。

【示範3】
資料：本季 EPS 2.1 元。
問題：全年 EPS 估計多少？幫我推估。
回答：資料中查無此資訊。

【資料】
{data}

【問題】{question}

請用繁體中文回答："""
    return llm.invoke(p).content


print("【陷阱題・few-shot】", few_shot_ask(push_q))
print()
print("【該答的題・few-shot 】", few_shot_ask(q))
print("【該答的題・沒有示範】", resp.content)
# 💡 最後兩行是重點：q 就是 F3 那題「這季毛利率多少？」，resp 是它當時的答案（沒有示範）。
#    同一題、同一份資料，差別只在「有沒有給示範」——比比看回答的長相

【陷阱題・few-shot】 資料中查無此資訊。



【該答的題・few-shot 】 38%
【該答的題・沒有示範】 第一季的毛利率是 38%。


### 🔑 few-shot 的副作用：它連「講話的樣子」都學走了

陷阱題一樣擋住。但**該答的那題**——

| | 回答 |
|---|---|
| **沒有示範** | 「第一季毛利率是 38%。」 |
| **給了 3 個示範** | 「**38%**」 |

**為什麼？因為示範裡的回答就是那麼短。** 它學走的不只是「什麼時候該拒答」，**還有「回答要長什麼樣」**。

> **🔑 帶走一句：示範餵進去的不只是「規則」，還有「語氣和格式」。你示範什麼，它就學什麼——連你沒打算教的也一起學。**
>
> 這**既是特性也是坑**：
> - **想要的時候**：這正是 few-shot 最強的用途——**用示範把輸出格式釘死**（要 JSON 就示範 JSON、要只回標籤就示範只回標籤）。
> - **不想要的時候**：你只想補強護欄，卻不小心把答案變成冷冰冰的兩個字 → **示範要連「你想要的回答風格」一起示範對。**

**💰 還有成本：** 每個示範都要**佔 token**，而且**每問一次就付一次**（示範跟著 prompt 一起送上去）。
這個護欄案例 **one-shot 就夠了**（G2-2 已經擋住）——**夠用就別再加。**

---

## 延伸 ・ 兩個 prompt 進階技巧：CoT / ToT（拿一道難題來看）

這兩招是**通用的 prompt 技巧**（不限 RAG）。這裡用一道 AI 研究常用的難題「**24 點**」來對比——給 4、9、10、13 四個數字，每個只用一次，用加減乘除湊出 24。

這題有多難？論文實測：GPT-4 **直接／CoT 只約 4% 答對**，改用 **ToT 有 74%**。`llm` 前面已接好（gemma4:cloud），下面直接用它。

### CoT（思維鏈）：叫它「一步一步想」

先看「直接要一個答案」——這種難題直接問是賭一把（可能湊對、也可能亂試或算錯）：

In [ ]:
g24 = "用 4、9、10、13 這四個數字，每個數字只能用一次，中間用加減乘除和括號，湊出 24。"

resp_direct = llm.invoke(g24 + " 只給我最終算式，不要寫任何過程。")
print(resp_direct.content)

(13 - 4) + 10 + 5 （抱歉，數字有誤，重新計算）

(13 + 10) - (9 - 4) = 18 (不對)

正確算式：
(13 - 9) * (10 - 4) = 24


同一題，加一句「請一步一步試」，讓它把嘗試過程攤開：

In [ ]:
resp_cot = llm.invoke(g24 + " 請一步一步嘗試不同組合，邊試邊檢查，最後給出正確算式。")
print(resp_cot.content)

這是一個有趣的數學挑戰。我們需要使用 4, 9, 10, 13 來湊出 24。

我將一步一步嘗試不同的組合路徑：

### 第一波嘗試：嘗試用 24 的倍數或接近值
*   **想法 1：** 嘗試 $13 + 11$。但剩下的 4, 9, 10 無法湊出 11。
*   **想法 2：** 嘗試 $10 + 14$。但 4, 9, 13 無法湊出 14。
*   **想法 3：** 嘗試 $4 \times 6$。我們有 4，但 9, 10, 13 無法湊出 6（例如 $13-10=3$, $9-3=6$）。
    *   **檢查：** $4 \times (9 - (13 - 10)) = 4 \times (9 - 3) = 4 \times 6 = 24$。

**這裡發現了一個可能的解法！讓我們驗證一遍：**
數字使用情況：
- 4 (使用了)
- 9 (使用了)
- 10 (使用了)
- 13 (使用了)
運算：$4 \times (9 - (13 - 10)) = 4 \times (9 - 3) = 4 \times 6 = 24$。

---

### 第二波嘗試：探索其他可能性（以確保是否有更簡潔的寫法）
*   **嘗試除法：** $24 \div 4 = 6$，或者 $24 \times \text{某數}$。
*   **嘗試大數減小數：**
    *   $13 + 10 + 9 - 4 = 28$ (太大了)
    *   $(13 \times 4) - (10 + 9) = 52 - 19 = 33$ (不對)
    *   $(10 \times 4) - (13 + 9) = 40 - 22 = 18$ (不對)
*   **嘗試分式組合：**
    *   $(13 - 4) \times 10 \div 9$ (不整除)

---

### 最終檢查與結論

經過嘗試，最直觀且正確的組合是將 13 和 10 相減得到 3，再用 9 減去 3 得到 6，最後乘以 4。

**正確算式為：**
$$4 \times (9 - (13 - 10)) = 24$$

或者去掉一層括號也可以寫成：
$$4 \times (9 - 13 + 10) = 24$$


### 🔑 CoT 帶走一句

CoT 把嘗試寫出來、比直接答更可靠，但它是「**一條線走到底**」——第一步走偏了就很難回頭。所以像 24 點這種要試很多條路的題，光靠 CoT 還是常卡住（論文只約 4%）。

### ToT（思維樹）：先分岔幾條路、評估，再挑最好的

同一題，改叫它「先列幾個第一步、各自評估有沒有希望、再沿最有希望的解到底」——這就是 ToT 的精神：

In [ ]:
resp_tot = llm.invoke(g24 + " 請先列出 3 種不同的第一步（各挑兩個數字做一種運算），逐一評估哪一步最有希望湊到 24，再沿最有希望的那條解到底，最後給出算式。")
print(resp_tot.content)

這是一個有趣的數學挑戰。我們的目標數字是 24，可用數字為 $\{4, 9, 10, 13\}$。

### 第一步：列出三種不同的嘗試方向

我將選擇兩個數字進行運算，觀察結果後評估後續湊到 24 的可能性：

1.  **方向 A：$13 - 9 = 4$**
    *   剩餘數字：$\{4, 10, 4\}$
    *   評估：得到了兩個 4 和一個 10。我想到了 $4 \times (10 - 4) = 24$。這個方向看起來非常有希望。

2.  **方向 B：$13 + 10 = 23$**
    *   剩餘數字：$\{4, 9, 23\}$
    *   評估：目前的數字 23 已經很接近 24，但剩下的 4 和 9 無法透過簡單運算產生 1（例如 $9-4=5$ 或 $9 \div 4=2.25$），很難湊到 24。

3.  **方向 C：$10 \times 4 = 40$**
    *   剩餘數字：$\{9, 13, 40\}$
    *   評估：數字變得較大，需要透過減法將 40 降至 24，也就是需要減去 16。但 $13 + 9 = 22$ 或 $13 - 9 = 4$，都無法精準產生 16。

---

### 第二步：沿著最有希望的「方向 A」推導

**第一步結果：** $(13 - 9) = 4$
**目前狀態：** 我們有三個數字 $\{4, 10, 4\}$ (其中一個 4 是由 $13-9$ 產生的)

**第二步思考：**
如何用 $4, 10, 4$ 湊出 24？
我發現 $10 - 4 = 6$，而 $4 \times 6 = 24$。

**第三步整合：**
將上述步驟合併：
$4 \times (10 - (13 - 9))$

---

### 第三步：最終算式驗算

$4 \times (10 - (13 - 9))$
$= 4 \times (10 - 4)$
$= 4 \times 6$
$= 24$

**最終算式：$4 \times (10 - (13 - 9)) = 24$**


### 🔑 ToT 帶走一句

ToT ＝ 先「多分支」評估再挑一條，比一條線的 CoT 更適合要搜尋、要試錯的題（24 點上論文 74% vs CoT 4%）。真正的 ToT 是讓模型**反覆搜尋很多次**，這裡是用一次 prompt 示意它的精神——認得名字、知道「先列幾條路再挑」就夠了。

> 🖥 這題是機率性的，你自己跑出來的過程可能跟上面不一樣（有時直接就湊對、有時要試很久）——這正是難題的本質。

---
## H・包成一個函式

之後每問一個問題，就是「檢索 → 組 prompt → 生成」這三步。包成函式，一行就能問。

---
## I ⭐ 重頭戲：為什麼不乾脆「全部貼進去」就好？

Lab 2 結尾丟了一個問題：**1000 份財報總不能全部貼進 prompt 吧？**

現在來真的量一次。同一個問題、三種做法，**盯著 token 用量看**。

### 📌 這一段下次上課會用到

「不給資料 / 全部貼進去 / 只檢索 2 塊」三種做法的 **token 用量對照**。

下一堂課（財報問答 RAG）會直接用到這三個數字，先看過會比較好跟。


### I1・做法 1：不給資料，直接問（就是 Lab 2 的 C1）

**預期輸出：** 它答不出來——「宏圖飲料」是虛構公司，模型訓練時根本沒看過。

In [28]:
target_q = "宏圖飲料 2026 Q1 的 EPS 是多少？"

r1 = llm.invoke(f"請用繁體中文回答：{target_q}")

print(r1.content)
print()
print("token 用量：", r1.usage_metadata)

目前沒有關於「宏圖飲料」2026 年第一季 EPS（每股盈餘）的公開數據。

原因如下：
1. **時間尚未到達**：現在尚未進入 2026 年，企業通常不會提前公布這麼遙遠的精確財務預測。
2. **缺乏公開資訊**：我的知識庫中沒有這家公司的特定財務預測報告。

如果您是在查看某份特定的分析報告或公司內部預測，建議您直接參閱該文件的「財務預測」或「分析師展望」章節。如果您是指某家上市公司的預估值，建議前往該公司的**投資人關係 (IR) 網站**或**證券交易所**查詢最新的分析師共識預測。

token 用量： {'input_tokens': 35, 'output_tokens': 163, 'total_tokens': 198}


### I2・做法 2（上）：把三份財報**全部**塞進模板，先看它多大

**預期輸出：** 組出來的 prompt 一千多字。

In [29]:
everything = "\n".join(raw)      # 💡 raw = C2 讀進來的三份財報原文，一個字都沒篩

msg_all = prompt.format_messages(ctx=everything, q=target_q)
# 💡 用新變數名 msg_all，不要覆蓋 target_q——下面 I3 還要拿 target_q 去檢索

print("整包塞進去的 prompt 共", len(msg_all[0].content), "個字")
print()
print(msg_all[0].content[:150], "...")

整包塞進去的 prompt 共 1773 個字

你是嚴謹的助理。只根據下面提供的資料回答問題，若資料中沒有答案就說「資料中查無此資訊」，不要自己編。

【資料】
【教學用虛構文件】宏圖飲料 2026 年第一季法人說明會 逐字稿摘錄
（本檔所有公司、人名、數字皆為虛構教學範例，非真實行情，請勿引用）

主持人：接下來請財務長說明第二季展望。

財務 ...


### I2・做法 2（下）：送出去

**預期輸出：** 答對 2.1 元。**注意看 `input_tokens`。**

In [30]:
r2 = llm.invoke(msg_all)

print(r2.content)
print()
print("token 用量：", r2.usage_metadata)

宏圖飲料 2026 年第一季的每股盈餘 EPS 為 2.1 元。

token 用量： {'input_tokens': 1361, 'output_tokens': 25, 'total_tokens': 1386}


### I3・做法 3（上）：RAG——先檢索，只把最相關的 2 塊塞進模板

**預期輸出：** 組出來的 prompt **短很多**。

In [31]:
r3_hits = vs.similarity_search(target_q, k=2)
r3_texts = []
for d in r3_hits:
    r3_texts.append(d.page_content)

msg_rag = prompt.format_messages(ctx="\n".join(r3_texts), q=target_q)

print("只塞 2 塊的 prompt 共", len(msg_rag[0].content), "個字")
print()
print(msg_rag[0].content[:150], "...")

只塞 2 塊的 prompt 共 631 個字

你是嚴謹的助理。只根據下面提供的資料回答問題，若資料中沒有答案就說「資料中查無此資訊」，不要自己編。

【資料】
【教學用虛構文件】宏圖飲料股份有限公司 2026 年第一季財報摘要
（本檔所有公司、人名、數字皆為虛構教學範例，非真實行情，請勿引用）

一、營運概況
宏圖飲料 2026 年第一季合併營 ...


### I3・做法 3（下）：送出去

**預期輸出：** 一樣答對 2.1 元，但 `input_tokens` 少很多。

In [32]:
r3 = llm.invoke(msg_rag)

print(r3.content)
print()
print("token 用量：", r3.usage_metadata)

宏圖飲料 2026 年第一季每股盈餘 EPS 為 2.1 元。

token 用量： {'input_tokens': 479, 'output_tokens': 24, 'total_tokens': 503}


### 🔑 把三個數字擺在一起看

| 做法 | input_tokens | 答對了嗎 |
|---|---|---|
| 1・不給資料 | 最少 | ❌ 答不出來 |
| 2・全部貼進去 | **最多** | ✅ |
| 3・RAG 檢索 2 塊 | **少很多** | ✅ |

做法 2 和做法 3 **答案一樣好**，但做法 2 花的 token 多好幾倍。

而我們的文件只有 **1,674 字、三個檔案**。

> **現在想像 1000 份財報**（大約 160 萬字）：
> - 做法 2 → **塞不進去**（模型的 context 有上限）；就算塞得進去，每問一次都要付 1000 份的錢。
> - 做法 3 → 不管幾份文件，每次都只送**最相關的 2 塊**。錢和速度**不會隨文件變多而暴漲**。
>
> **這就是為什麼要有檢索這一站。** RAG 不只是「答得準」，是「**答得準，而且付得起**」。

### 📝 小作業 B

把 `k` 從 2 改成 5，問同一個問題，看看：

1. 答案有變好嗎？
2. `input_tokens` 變多少？

<details><summary>📖 做完再看：參考解（參考解不只一種）</summary>

```python
k5_hits = vs.similarity_search(target_q, k=5)
k5_texts = []
for d in k5_hits:
    k5_texts.append(d.page_content)

k5 = llm.invoke(prompt.format_messages(ctx="\n".join(k5_texts), q=target_q))
print(k5.content)
print("token 用量：", k5.usage_metadata)
```

**結論：** 答案沒有變好（本來就答對了），但 token 明顯變多。

**k 不是越大越好**——k 太大會撈進不相關的塊，除了多花錢，還可能把答案帶偏；k 太小則怕漏掉關鍵段落。一般 **k=2~4** 起步。
</details>

In [33]:
# 📝 小作業 B 參考解（參考解不只一種）

# 把 k 從 2 改成 5，問同一個問題
k5_hits = vs.similarity_search(target_q, k=5)
k5_texts = []
for d in k5_hits:
    k5_texts.append(d.page_content)

k5 = llm.invoke(prompt.format_messages(ctx="\n".join(k5_texts), q=target_q))
print(k5.content)
print("token 用量：", k5.usage_metadata)   # 💡 對照 k=2 時的 input_tokens，會明顯變多

# 結論：答案沒有變好（本來就答對了），但 token 明顯變多。
#       k 不是越大越好——太大會撈進不相關的塊，多花錢還可能把答案帶偏；太小怕漏關鍵段落。
#       一般 k=2~4 起步。

宏圖飲料 2026 年第一季每股盈餘 EPS 為 2.1 元。
token 用量： {'input_tokens': 940, 'output_tokens': 24, 'total_tokens': 964}


---
### 🛟 Backup：雲端不通時，改用本機模型（平常不用跑）

In [34]:
# 平常不用跑這格。雲端不通（401 / 連線失敗 / 額度用完）時，把下面兩行的 # 拿掉再跑，
# 後面所有格子就會改用本機模型。品質會降，但流程不會斷。

# from langchain_ollama import ChatOllama
# llm = ChatOllama(model="llama3.2:3b")     # 先在終端機下：ollama pull llama3.2:3b

---
## 🎓 你今天做出了什麼

```
你的文件 → 切塊 → embedding → 向量庫                      （建索引・做一次）
問題 → 撈最像的 k 塊 → 組 prompt → LLM → 有依據的答案      （查詢・每次跑）
```

**上午的檢索 ＋ 下午的生成 ＝ RAG。** 這就是「從關鍵字搜尋到 RAG」這條主線的終點。